# Deep Technical Report: ProPicker Architecture and Methodology

## 1. Executive Summary

ProPicker is a promptable, deep learning-based 3D segmentation model designed for particle picking in Cryogenic Electron Tomography (cryo-ET). Unlike traditional methods that rely purely on exhaustive template matching or static object detection models trained on fixed classes, ProPicker utilizes a "promptable" architecture. It takes a single example (a prompt) of a target particle and a tomogram volume as input to generate a 3D segmentation mask. This allows for zero-shot detection of unseen particles and highly data-efficient fine-tuning.

## 2. Problem Domain: Cryo-ET Particle Picking

Particle picking is the task of localizing specific macromolecules within 3D tomograms. Formally, the system takes a tomogram $x \in \mathbb{R}^{n \times n \times n}$ and a prompt $p \in \mathbb{R}^{m \times m \times m}$ as input. The objective is to produce a finite set of coordinates $\mathcal{C} \subset \{1, ..., n\}^3$ representing the centers of the particles.

This task is treated as a 3D object detection problem characterized by:

- **Low Signal-to-Noise Ratio (SNR)**: Tomograms are extremely noisy due to dose limitations in electron microscopy.
- **Crowded Environments**: Cells contain diverse, densely packed proteins.
- **Computational Load**: Tomograms are massive (e.g., $200 \times 1000 \times 1000$ voxels), making sliding-window template matching computationally expensive.

## 3. System Architecture

ProPicker decouples the feature extraction of the target particle from the segmentation of the volume. The architecture consists of two primary modules: a Prompt Encoder ($\epsilon$) and a Conditional Segmentation Network ($\mathcal{S}$).

### 3.1. The Prompt Encoder

The prompt encoder is responsible for generating a latent representation of the target particle.

- **Definition**: The encoder is defined as a function $\epsilon: \mathbb{R}^{m \times m \times m} \rightarrow \mathbb{R}^d$.
- **Input**: A small 3D crop (sub-tomogram) containing the particle of interest (the prompt), typically sized $37 \times 37 \times 37$ voxels.
- **Model**: ProPicker utilizes the TomoTwin encoder.
- **Output**: A feature vector $\epsilon(p) \in \mathbb{R}^{d}$, where $d=32$ in the implementation.
- **Function**: This vector encodes the structural identity of the particle. During training and inference, this encoder is typically frozen, ensuring consistent embeddings.

### 3.2. Conditional Segmentation Model (3D U-Net)

The core of ProPicker is a modified 3D U-Net that performs voxel-wise binary classification.

- **Definition**: The model is a function $\mathcal{S}: \mathbb{R}^{n \times n \times n} \times \mathbb{R}^d \rightarrow \mathbb{R}^{n \times n \times n}$.
- **Output**: It produces an output map $y = \mathcal{S}(x; \epsilon(p))$, where theoretically $y \in \{0, 1\}^{n \times n \times n}$ (implemented as a probability map via a sigmoid activation).
- **Architecture Base**: A 3D U-Net following the encoder-decoder structure, significantly scaled up from standard implementations (e.g., DeepETPicker).
- **Parameters**: Approximately $100$ million trainable parameters.
- **Depth**: 5 convolution-based spatial downsampling layers (Encoder) and 5 spatial upsampling layers (Decoder).
- **Channel Widths**: The layers process volumes with $108$, $216$, $432$, and $864$ channels respectively.
- **Normalization**: Instance Normalization is used instead of Batch Normalization. This is critical for processing large 3D volumes where batch sizes must be kept small due to memory constraints.

### 3.3. Conditioning Mechanism (FiLM)

To steer the generic U-Net to detect only the particle specified by the prompt, ProPicker uses Feature-wise Linear Modulation (FiLM).

- **Location**: FiLM layers are applied to each of the 5 spatial upsampling layers in the Decoder.
- **Mechanism**:
  - Let $c$ be the number of channels in an intermediate feature map. The 32-dim prompt vector $\epsilon(p)$ is projected using two learnable matrices $A, B \in \mathbb{R}^{c \times d}$.
  - This generates channel-specific modulation parameters: a slope $(A\epsilon(p))_k$ and an intercept $(B\epsilon(p))_k$ for each channel $k$.
  - For a feature map $F$ with $c$ channels, the modulation is applied as:

$$\text{FiLM}(F_k) = (A\epsilon(p))_k \cdot F_k + (B\epsilon(p))_k$$

This effectively highlights features in the activation maps that correlate with the geometric structure of the prompt while suppressing irrelevant features.

## 4. Operational Workflow

### 4.1. Inference (Prompt-Based Picking)

1. **Prompt Selection**: A user selects a sub-tomogram containing the target particle.
2. **Embedding**: The Prompt Encoder converts this into the vector $\epsilon(p)$.
3. **Sliding Window Segmentation**:
   - Since tomograms are too large to process at once, a window (default size $64 \times 64 \times 64$) slides across the volume.
   - **Stride**: A large stride (default $32$ voxels) is used. Unlike template matching which requires single-voxel strides (e.g., $s=1$ or $s=2$), the segmentation approach allows for larger steps, significantly increasing speed.
   - Overlapping predictions are averaged to produce a full-volume probability map.
4. **Coordinate Extraction**:
   - **ProPicker-C (Clustering)**: The segmentation mask $y$ is binarized. Connected components are analyzed to find centroids. Clusters falling outside specific size thresholds are discarded.
   - **ProPicker-TM (Template Matching)**: Used for difficult, crowded scenes. The segmentation mask acts as a region-of-interest filter. A template matcher is run only inside the masked regions to separate overlapping particles.

### 4.2. Training Methodology

- **Dataset**: Trained on 85 synthetic tomograms containing 121 unique protein types.
- **Procedure**:
  - **Input**: Batches of 8 sub-tomograms ($64^3$).
  - **Prompts**: 8 random prompts are sampled per batch.
  - **Loss**: Binary Cross Entropy (BCE) is calculated between the predicted mask and the ground truth mask for the specific prompted particle.
  - **Note**: The authors found BCE superior to Dice loss for this specific architecture, particularly in avoiding merged masks in crowded environments.

### 4.3. Fine-Tuning

When zero-shot performance is insufficient, ProPicker supports "Few-Shot" fine-tuning.

- **Frozen Components**: The Prompt Encoder weights and the Prompt vector $\epsilon(p)$ remain fixed.
- **Trainable Components**: The 3D U-Net weights $\mathcal{S}$ are updated.
- **Data Efficiency**: Experiments show that fine-tuning on as little as $25\%$ of a single tomogram can boost F1 scores significantly (e.g., from $\approx 0.15$ to $\approx 0.70$), outperforming models like DeepETPicker that require training from scratch.

## 5. Performance Characteristics

- **Generalization**: Capable of detecting unseen particles in synthetic data and diverse real-world datasets (e.g., DS-10440, EMPIAR-10988).
- **Speed**: By leveraging 3D segmentation with large strides, it achieves higher throughput (tomograms per hour) compared to dense template matching methods.
- **Limitations**: Like many deep learning methods, zero-shot performance can degrade on very small or low-contrast particles (e.g., beta-amylase) without fine-tuning. It relies on the existence of at least one clear example to serve as a prompt.
